# mask meta：G × 分块一阶内积 金标准
**回答的问题：G = |∂L_task/∂x₀| 作为掩码选址代理有没有价值。**

空间版"ut × meta-grad"：真实量是**分块一阶内积** IP_i = Σ_{p∈i} g_task(p)·g_den(p)
（输入空间分解，Σ_i IP_i = ⟨∂L_task/∂x₀, ∂L_den/∂x₀⟩，与论文 3.3 的一阶展开逐块对应）；
代理量是 G 的分块均值（与在线 MaskSelector 同口径）。逐图 64 块的 Spearman/Pearson 越高，
G 选址越可信。

每个 ckpt 出一张四联图（原图 | IP 正部热力 | G 热力 | 现采掩码 m），跨 ckpt 汇总相关性演化；
测量结果缓存 npz，复画免重测。CRN：跨 ckpt 同图、同 t、同噪声，严格可比。

In [ ]:
import os, sys
import torch

# ---- 先在 notebook 所在目录（visual/）下把两个路径都算成绝对路径 ----
VISUAL = os.path.abspath('.')                                  # 内核启动时 cwd 就是 visual/
MMSEG  = os.path.abspath('../tasks/Segmentation/mmsegmentation')

os.chdir(MMSEG)
for p in [MMSEG, os.path.join(MMSEG, 'eden'), VISUAL]:
    if p not in sys.path:
        sys.path.insert(0, p)

# eden 包在服务器上部署名为 mydiffae：本地别名兼容
import eden, eden.diffae_mixin
sys.modules.setdefault('mydiffae', eden)
sys.modules.setdefault('mydiffae.diffae_mixin', eden.diffae_mixin)

# 注册 adapter（FCNWithDiffAE）；已注册过则跳过
try:
    import mmseg.models.segmentors.eden_adapter_fcn  # noqa
except KeyError:
    pass

import probe_core as pc

In [ ]:
# ---- 配置：改成你的 run 目录 ----
# CFG 用 run 目录里 train 时 dump 的完整 config；CKPTS 按 iter 排序。
# 旧 run 兼容：build_model 会自动剔除 dump config 里已退役的机制键，旧 ckpt 直接可测；
# UNet 仪器的旧 run 需把 UNET_FIX 改为 dict(denoiser_type='unet')。
# 注意：掩码现采走当前代码的 MaskSelector——旧 run 的蒙块配置键若已退役，
# 掩码面板反映的是【当前】选址逻辑（ratio/alpha 用新默认或被保留的旧值）。
import glob, re
RUN_DIR = 'work_dirs/eden/fcn_diffae'
CFG = sorted(glob.glob(os.path.join(RUN_DIR, '*.py')))[0]   # run 目录里 dump 的 config
UNET_FIX = None   # UNet 仪器的旧 run 改为 dict(denoiser_type='unet')；DiT run 保持 None
CKPTS = sorted(glob.glob(f'{RUN_DIR}/iter_*.pth'),
               key=lambda p: int(re.findall(r'\d+', os.path.basename(p))[-1]))
OUT = 'work_dirs/offline_probe/mask_meta'
NUM_IMAGES = 4        # 展示的图数（val 前 N 张，跨 ckpt 固定）
N_T = 8               # IP 测量的 t 点数（训练网格等距）
N_NOISE = 2           # 每 t 的 CRN 噪声数（ε 臂方差≈0，2 个足够）
SEED = 0
os.makedirs(OUT, exist_ok=True)

In [ ]:
# 数据只取一次，所有 ckpt 共用（同图同分布，跨 ckpt 可比）
model, cfg = pc.build_model(CFG, cfg_overrides=UNET_FIX)   # 先建一次只为取数
batch = pc.build_val_batch(model, cfg, NUM_IMAGES)
del model; torch.cuda.empty_cache()

In [ ]:
rows = []
for ckpt in CKPTS:
    model, _ = pc.build_model(CFG, ckpt, cfg_overrides=UNET_FIX)
    tag = os.path.splitext(os.path.basename(ckpt))[0]
    res = pc.measure_mask_meta(model, batch, n_t=N_T, n_noise=N_NOISE, seed=SEED)
    # 缓存（复画免重测）
    import numpy as np
    np.savez(os.path.join(OUT, f'{tag}_cache.npz'),
             g_block=res['g_block'].cpu().numpy(),
             ip_block=res['ip_block'].cpu().numpy(),
             mask=res['mask'].cpu().numpy(),
             rho_s=res['rho_s'], rho_p=res['rho_p'], t=res['t'])
    fig = pc.plot_mask_meta(res, title=tag,
                            save_path=os.path.join(OUT, f"{tag}_mask_meta.png"))
    rows.append((tag, res['rho_s'], res['rho_p']))
    print(f"{tag}: Spearman ρ mean={res['rho_s'].mean():.3f} "
          f"±{res['rho_s'].std():.3f} | Pearson r mean={res['rho_p'].mean():.3f} "
          f"±{res['rho_p'].std():.3f}  (per-image: {[f'{v:.2f}' for v in res['rho_s']]})")
    del model; torch.cuda.empty_cache()
fig  # 显示最后一张

In [ ]:
# 跨 ckpt 汇总：G ~ 分块一阶内积 的相关性随训练演化（均值 ± 跨图 std）
import numpy as np
import matplotlib.pyplot as plt
if len(rows) > 1:
    tags = [r[0] for r in rows]
    rs = np.array([r[1] for r in rows])   # [n_ckpt, n_img]
    rp = np.array([r[2] for r in rows])
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    for vals, name, c in [(rs, 'Spearman ρ', 'tab:blue'), (rp, 'Pearson r', 'tab:orange')]:
        m, s = vals.mean(1), vals.std(1)
        ax.plot(tags, m, 'o-', color=c, label=name)
        ax.fill_between(tags, m - s, m + s, color=c, alpha=0.2)
    ax.axhline(0, color='k', ls='--', lw=0.8)
    ax.set_ylim(-1, 1); ax.grid(alpha=0.3); ax.legend(fontsize=9)
    ax.set_title('G ~ block-wise meta IP across checkpoints (mean ± std over images)')
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
    fig.savefig(os.path.join(OUT, 'summary.png'), dpi=150, bbox_inches='tight')
    fig